In [1]:
import tkinter as tk
from tkinter import messagebox
import json
import os

class FlashcardApp:
    def __init__(self, root):
        # Инициализация главного окна
        self.root = root
        self.root.title("Флэш-карточки v1.0")
        self.root.geometry("600x400")

        # Данные приложения
        self.cards = []
        self.current_card_index = 0

        # Создание виджетов интерфейса
        self.create_widgets()
        # Загрузка данных
        self.load_data()

    def create_widgets(self):
        """Создаёт все элементы интерфейса"""
        # Метка для отображения термина
        self.term_label = tk.Label(self.root, text="", font=("Arial", 24), wraplength=500)
        self.term_label.pack(pady=50)

        # Кнопка для показа определения
        self.show_answer_btn = tk.Button(self.root, text="Показать определение", command=self.show_definition, font=("Arial", 14))
        self.show_answer_btn.pack(pady=10)

        # Текстовое поле для отображения определения (изначально скрыто)
        self.definition_text = tk.Text(self.root, height=6, width=60, font=("Arial", 12), state='disabled')
        self.definition_text.pack(pady=10)

        # Фрейм для кнопок навигации
        button_frame = tk.Frame(self.root)
        button_frame.pack(pady=20)

        self.knew_btn = tk.Button(button_frame, text="Знаю", command=self.mark_known, font=("Arial", 12), bg="lightgreen")
        self.knew_btn.pack(side=tk.LEFT, padx=10)

        self.forgot_btn = tk.Button(button_frame, text="Забыл", command=self.mark_forgotten, font=("Arial", 12), bg="lightcoral")
        self.forgot_btn.pack(side=tk.LEFT, padx=10)

        self.next_btn = tk.Button(button_frame, text="Следующая", command=self.next_card, font=("Arial", 12))
        self.next_btn.pack(side=tk.LEFT, padx=10)

    def load_data(self):
        """Загружает карточки из файла"""
        try:
            with open('flashcards_data.json', 'r', encoding='utf-8') as f:
                self.cards = json.load(f)
        except FileNotFoundError:
            # Если файла нет, создаём пример данных
            self.cards = [
                {"term": "PR-кривая", "definition": "График, показывающий компромисс между точностью и полнотой модели.", "errors": 0},
                {"term": "SMOTE", "definition": "Метод синтетической передискретизации для борьбы с дисбалансом классов.", "errors": 0}
            ]
            self.save_data()

        if self.cards:
            self.show_card(0)

    def save_data(self):
        """Сохраняет карточки в файл"""
        with open('flashcards_data.json', 'w', encoding='utf-8') as f:
            json.dump(self.cards, f, ensure_ascii=False, indent=2)

    def show_card(self, index):
        """Показывает карточку по индексу"""
        self.current_card_index = index
        card = self.cards[index]

        # Обновляем интерфейс
        self.term_label.config(text=f"Термин: {card['term']}")
        self.definition_text.config(state='normal')
        self.definition_text.delete(1.0, tk.END)
        self.definition_text.config(state='disabled')
        self.show_answer_btn.config(state='normal')

    def show_definition(self):
        """Показывает определение текущей карточки"""
        card = self.cards[self.current_card_index]
        self.definition_text.config(state='normal')
        self.definition_text.delete(1.0, tk.END)
        self.definition_text.insert(1.0, f"Определение:\n{card['definition']}")
        self.definition_text.config(state='disabled')
        self.show_answer_btn.config(state='disabled')

    def mark_known(self):
        """Отмечает, что термин известен"""
        if self.cards:
            self.next_card()

    def mark_forgotten(self):
        """Отмечает, что термин забыт, и увеличивает счётчик ошибок"""
        if self.cards:
            self.cards[self.current_card_index]['errors'] += 1
            self.save_data()
            self.next_card()

    def next_card(self):
        """Переходит к следующей карточке"""
        if not self.cards:
            return

        # Простая логика: показываем карточки с наибольшим числом ошибок
        self.cards.sort(key=lambda x: x['errors'], reverse=True)
        next_index = (self.current_card_index + 1) % len(self.cards)
        self.show_card(next_index)

# Точка входа в приложение
if __name__ == "__main__":
    root = tk.Tk()
    app = FlashcardApp(root)
    root.mainloop()